# Phase 2 — Step 8: Apply Model to 22 Variants and Rank

**Models used (from Step 6 grid search with REAL CADD scores):**
- RF : n_estimators=200, max_depth=6, AUC=0.9052
- XGB: n_estimators=100, max_depth=4, AUC=0.9056

**What this notebook does:**
1. Applies best RF + XGB to all 22 Phase 1 candidates
2. Predicts pathogenicity risk scores (0.0 — 1.0)
3. Assigns tiers: HIGH ≥0.75 | MODERATE 0.50-0.74 | LOW <0.50
4. Consensus score = mean(RF, XGB)
5. Ranked bar chart, scatter, CADD vs risk plot
6. Identifies Phase B shortlist
7. Saves final CSV to Drive

**Run cells one by one**

In [ ]:
!pip install scikit-learn xgboost pandas numpy matplotlib seaborn joblib -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns
import joblib
import warnings
import os
warnings.filterwarnings('ignore')
os.makedirs('figures_step8', exist_ok=True)
print('Libraries ready')

In [ ]:
from google.colab import drive
import joblib, os
drive.mount('/content/drive')

RF_PATH  = '/content/drive/My Drive/best_rf_model_step6.pkl'
XGB_PATH = '/content/drive/My Drive/best_xgb_model_step6.pkl'
IMP_PATH = '/content/drive/My Drive/imputer_step6.pkl'

for p in [RF_PATH, XGB_PATH, IMP_PATH]:
    status = '✅' if os.path.exists(p) else '❌'
    print(f'{status} {os.path.basename(p)}')

best_rf  = joblib.load(RF_PATH)
best_xgb = joblib.load(XGB_PATH)
imputer  = joblib.load(IMP_PATH)
print('Models loaded')

In [ ]:
import pandas as pd
import numpy as np

# All 22 Phase 1 candidate variants — full annotation table
variants_22 = pd.DataFrame({
    'gene'        : ['HP1BP3','AGAP4','KDM5A','C12orf40','PAPOLA','RTF1','USP7',
                     'EPG5','PIGF','SPICE1','SPICE1','SLC2A2','ZNF141','TRIML2',
                     'TRIML2','SREK1','SREK1','RAD50','RARS2','TNRC18','DBF4','LRRCC1'],
    'chrom'       : ['chr1','chr10','chr12','chr12','chr14','chr15','chr16',
                     'chr18','chr2','chr3','chr3','chr3','chr4','chr4',
                     'chr4','chr5','chr5','chr5','chr6','chr7','chr7','chr8'],
    'pos'         : [21072082,46342157,394677,40044027,97022168,41709326,8988686,
                     43469840,46839487,113218285,113218285,170727846,367275,189020278,
                     189020278,65473406,65473406,131927550,88224021,5385191,87516629,86022336],
    'variant_type': ['missense_variant','missense_variant','missense_variant',
                     'splice_region_variant','splice_acceptor_variant',
                     'missense_variant','missense_variant','missense_variant',
                     'splice_region_variant','splice_donor_variant','splice_donor_variant',
                     'missense_variant','missense_variant','missense_variant',
                     'missense_variant','missense_variant','missense_variant',
                     'splice_region_variant','missense_variant','splice_donor_variant',
                     'splice_region_variant','splice_region_variant'],
    'impact'      : ['MODERATE','MODERATE','MODERATE','MODERATE','HIGH',
                     'MODERATE','MODERATE','MODERATE','MODERATE','HIGH','HIGH',
                     'MODERATE','MODERATE','MODERATE','MODERATE','MODERATE',
                     'MODERATE','MODERATE','MODERATE','HIGH','MODERATE','MODERATE'],
    'cadd_phred'  : [23.5,4.63,13.88,3.54,2.71,22.0,24.7,21.1,0.43,
                    25.1,25.8,0.0,1.66,9.62,19.43,23.7,23.0,1.07,
                    1.45,22.2,0.57,5.45],
    'gerp_rs'     : [5.87,1.34,4.56,-0.78,-0.58,3.63,3.50,3.30,2.94,
                    5.94,5.94,3.08,1.24,3.00,3.00,5.45,5.45,-3.36,
                    -3.08,5.19,1.38,0.77],
    'sift_score'  : [0.15,0.03,0.06,np.nan,np.nan,0.37,0.03,0.00,np.nan,
                    np.nan,np.nan,1.00,0.69,0.01,0.03,0.00,0.03,np.nan,
                    0.03,np.nan,np.nan,np.nan],
    'polyphen_score':[0.91,0.02,0.02,np.nan,np.nan,0.01,0.01,0.55,np.nan,
                     np.nan,np.nan,0.02,0.00,0.28,0.02,0.58,0.02,np.nan,
                     0.02,np.nan,np.nan,np.nan],
    'spliceai_score':[0.01,0.01,0.01,0.04,0.01,0.01,0.01,0.01,0.01,
                     0.91,0.91,0.01,0.01,0.01,0.01,0.01,0.01,0.01,
                     0.01,1.00,0.02,0.01],
    'clinvar_sig' : ['Pathogenic','Likely_Pathogenic','Likely_Pathogenic','N/A',
                     'Likely_Pathogenic','Pathogenic','Pathogenic',
                     'Pathogenic/Likely_Pathogenic','Pathogenic','Pathogenic',
                     'Pathogenic','Likely_Pathogenic','Pathogenic/Likely_Pathogenic',
                     'Pathogenic','Pathogenic','Pathogenic/Likely_Pathogenic',
                     'Pathogenic/Likely_Pathogenic','Pathogenic/Likely_Pathogenic',
                     'Pathogenic/Likely_Pathogenic','Pathogenic','Pathogenic','Pathogenic'],
    'is_novel'    : [1]*22,
    'in_controls' : [1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
})

print(f'22-variant table built: {len(variants_22)} variants')
print(f'Impact: {variants_22["impact"].value_counts().to_dict()}')

In [ ]:
# Predict pathogenicity
X_22 = imputer.transform(variants_22[['cadd_phred']].values)
rf_probs  = best_rf.predict_proba(X_22)[:, 1]
xgb_probs = best_xgb.predict_proba(X_22)[:, 1]
consensus = (rf_probs + xgb_probs) / 2.0

def tier(p):
    return 'HIGH' if p>=0.75 else 'MODERATE' if p>=0.50 else 'LOW'

variants_22['rf_prob']    = np.round(rf_probs, 4)
variants_22['xgb_prob']   = np.round(xgb_probs, 4)
variants_22['consensus']  = np.round(consensus, 4)
variants_22['rf_tier']    = [tier(p) for p in rf_probs]
variants_22['xgb_tier']   = [tier(p) for p in xgb_probs]
variants_22['cons_tier']  = [tier(p) for p in consensus]
variants_22['model_agree']= (variants_22['rf_tier']==variants_22['xgb_tier'])

variants_22.sort_values('consensus', ascending=False, inplace=True)
variants_22.reset_index(drop=True, inplace=True)

print('PREDICTIONS (sorted by consensus):')
print(variants_22[['gene','impact','cadd_phred','rf_prob','xgb_prob',
                    'consensus','cons_tier','model_agree']].to_string(index=False))
print()
print('Tier distribution:')
print(variants_22['cons_tier'].value_counts().to_string())

In [ ]:
# Ranked bar chart
tier_colors = {'HIGH':'#C0392B','MODERATE':'#E67E22','LOW':'#27AE60'}
bar_colors  = [tier_colors[t] for t in variants_22['cons_tier']]

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(variants_22)), variants_22['consensus'],
               color=bar_colors, height=0.65, edgecolor='white')

for i, (bar, row) in enumerate(zip(bars, variants_22.itertuples())):
    star = ' ★' if row.impact == 'HIGH' else ''
    ax.text(-0.005, bar.get_y()+bar.get_height()/2,
            f'{row.gene}{star}', va='center', ha='right', fontsize=9,
            fontweight='bold' if row.impact=='HIGH' else 'normal')
    ax.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
            f'{row.consensus:.3f}', va='center', fontsize=8)
    if row.consensus > 0.10:
        ax.text(bar.get_width()*0.5, bar.get_y()+bar.get_height()/2,
                f'CADD={row.cadd_phred:.1f}', va='center', ha='center',
                fontsize=7, color='white', fontweight='bold')

ax.axvline(0.75, color='#C0392B', ls='--', lw=1.5, alpha=0.8)
ax.axvline(0.50, color='#E67E22', ls='--', lw=1.5, alpha=0.8)
ax.set_yticks([])
ax.set_xlabel('Consensus Pathogenicity Score', fontsize=11)
ax.set_title('Ranked Pathogenicity Risk Scores — 22 Phase 1 Candidates\n'
             'RF + XGBoost Consensus | Real CADD Scores | AUC=0.904',
             fontsize=12, fontweight='bold')
ax.set_xlim(-0.30, 1.05)
legend_els = [mpatches.Patch(fc='#C0392B',label='HIGH (≥0.75)'),
              mpatches.Patch(fc='#E67E22',label='MODERATE (0.50-0.74)'),
              mpatches.Patch(fc='#27AE60',label='LOW (<0.50)'),
              mlines.Line2D([0],[0],marker='',color='black',
                            label='★ = HIGH SnpEff impact')]
ax.legend(handles=legend_els, fontsize=9, loc='lower right')
ax.spines[['top','right','left']].set_visible(False)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('figures_step8/ranked_pathogenicity_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Ranked bar chart saved')

In [ ]:
# RF vs XGB scatter
fig, ax = plt.subplots(figsize=(9, 8))
tier_colors = {'HIGH':'#C0392B','MODERATE':'#E67E22','LOW':'#27AE60'}

for _, row in variants_22.iterrows():
    color  = tier_colors[row['cons_tier']]
    marker = 'D' if row['impact']=='HIGH' else 'o'
    ax.scatter(row['rf_prob'], row['xgb_prob'], c=color,
               marker=marker, s=120, edgecolors='white', zorder=3)
    ax.annotate(row['gene'], (row['rf_prob'], row['xgb_prob']),
                textcoords='offset points', xytext=(5,4), fontsize=8,
                fontweight='bold' if row['impact']=='HIGH' else 'normal')

ax.plot([0,1],[0,1], color='#AAAAAA', ls='--', lw=1.5, label='Perfect agreement')
ax.axhline(0.75, color='#C0392B', ls=':', lw=1.2, alpha=0.5)
ax.axhline(0.50, color='#E67E22', ls=':', lw=1.2, alpha=0.5)
ax.axvline(0.75, color='#C0392B', ls=':', lw=1.2, alpha=0.5)
ax.axvline(0.50, color='#E67E22', ls=':', lw=1.2, alpha=0.5)

r = np.corrcoef(variants_22['rf_prob'], variants_22['xgb_prob'])[0,1]
ax.text(0.05, 0.93, f'Pearson r = {r:.4f}', transform=ax.transAxes,
        fontsize=11, fontweight='bold')

legend_els = [mpatches.Patch(fc='#C0392B',label='HIGH risk'),
              mpatches.Patch(fc='#E67E22',label='MODERATE risk'),
              mpatches.Patch(fc='#27AE60',label='LOW risk'),
              mlines.Line2D([0],[0],marker='D',color='w',
                            markerfacecolor='grey',ms=9,label='HIGH impact'),
              mlines.Line2D([0],[0],marker='o',color='w',
                            markerfacecolor='grey',ms=9,label='MODERATE impact')]
ax.legend(handles=legend_els, fontsize=9)
ax.set_xlabel('RF Pathogenicity Probability', fontsize=12)
ax.set_ylabel('XGBoost Pathogenicity Probability', fontsize=12)
ax.set_title('RF vs XGBoost Predictions — 22 Candidates', fontsize=12, fontweight='bold')
ax.set_xlim(-0.02,1.05); ax.set_ylim(-0.02,1.05)
ax.grid(True, alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step8/rf_vs_xgb_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'RF vs XGB scatter saved (r={r:.4f})')

In [ ]:
# CADD vs Consensus score
fig, ax = plt.subplots(figsize=(11, 7))
tier_colors = {'HIGH':'#C0392B','MODERATE':'#E67E22','LOW':'#27AE60'}

for _, row in variants_22.iterrows():
    color  = tier_colors[row['cons_tier']]
    marker = 'D' if row['impact']=='HIGH' else 'o'
    ax.scatter(row['cadd_phred'], row['consensus'], c=color,
               marker=marker, s=140, edgecolors='white', zorder=3)
    ax.annotate(row['gene'], (row['cadd_phred'], row['consensus']),
                textcoords='offset points', xytext=(5,4), fontsize=8.5,
                fontweight='bold' if row['impact']=='HIGH' else 'normal')

ax.axhline(0.75, color='#C0392B', ls='--', lw=1.5, alpha=0.8)
ax.axhline(0.50, color='#E67E22', ls='--', lw=1.5, alpha=0.8)
ax.axvline(15,   color='grey',    ls=':',  lw=1.2, alpha=0.6)
ax.axvline(20,   color='grey',    ls='-',  lw=1.2, alpha=0.4)

legend_els = [mpatches.Patch(fc='#C0392B',label='HIGH risk'),
              mpatches.Patch(fc='#E67E22',label='MODERATE risk'),
              mpatches.Patch(fc='#27AE60',label='LOW risk'),
              mlines.Line2D([0],[0],marker='D',color='w',
                            markerfacecolor='grey',ms=9,label='HIGH impact (splice)'),
              mlines.Line2D([0],[0],color='#C0392B',ls='--',label='HIGH threshold (0.75)'),
              mlines.Line2D([0],[0],color='#E67E22',ls='--',label='MOD threshold (0.50)')]
ax.legend(handles=legend_els, fontsize=9, loc='lower right')
ax.set_xlabel('CADD Phred Score', fontsize=12)
ax.set_ylabel('Consensus Pathogenicity Score', fontsize=12)
ax.set_title('CADD Score vs Consensus Pathogenicity\n22 Phase 1 Candidates | Real CADD Scores',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures_step8/cadd_vs_pathogenicity.png', dpi=150, bbox_inches='tight')
plt.show()
print('CADD vs pathogenicity scatter saved')

In [ ]:
# Final ranked table
print('FINAL RANKED TABLE')
print('='*100)
for i, row in variants_22.iterrows():
    tier_sym = {'HIGH':'🔴','MODERATE':'🟠','LOW':'🟢'}[row['cons_tier']]
    agree = '✓' if row['model_agree'] else '~'
    print(f'{i+1:<4} {row.gene:<10} {row.impact:<10} '
          f'CADD={row.cadd_phred:<6.1f} '
          f'RF={row.rf_prob:.3f} XGB={row.xgb_prob:.3f} '
          f'Cons={row.consensus:.3f} {tier_sym} {row.cons_tier:<10} {agree}')
print()
print('Tier distribution:')
print(variants_22['cons_tier'].value_counts().to_string())
print()
print('Model agreement:', variants_22['model_agree'].sum(), '/ 22')

In [ ]:
# Phase B shortlist
print('PHASE B SHORTLIST — HIGH PRIORITY VARIANTS')
print('='*70)

high = variants_22[variants_22['cons_tier']=='HIGH']
high_impact_not_top = variants_22[
    (variants_22['impact']=='HIGH') &
    (variants_22['cons_tier']!='HIGH')
]

print(f'Consensus HIGH tier: {len(high)} variants')
if len(high) > 0:
    print(high[['gene','cadd_phred','consensus','clinvar_sig']].to_string(index=False))

print(f'\nHIGH SnpEff impact not in HIGH tier: {len(high_impact_not_top)}')
if len(high_impact_not_top) > 0:
    print(high_impact_not_top[['gene','cadd_phred','consensus','spliceai_score']].to_string(index=False))

print()
print('NOTE: Splice variants (SPICE1, PAPOLA, TNRC18) may score LOW/MODERATE')
print('because CADD does not score splice sites well.')
print('SpliceAI scores for these are HIGH (0.91, 1.00) — carry forward to Phase B.')

In [ ]:
import shutil
from google.colab import files

# Save to Drive
drive_path = '/content/drive/My Drive/step8_22variants_ranked_final.csv'
variants_22.to_csv(drive_path, index=False)
print(f'Saved to Drive: {drive_path}')

figs = sorted(os.listdir('figures_step8'))
print(f'Figures ({len(figs)}):')
for f in figs:
    print(f'  {f}')

# Zip and download
shutil.make_archive('step8_outputs', 'zip', 'figures_step8')
files.download('step8_outputs.zip')
files.download(drive_path)
print('Downloaded step8_outputs.zip and CSV')

print()
print('PHASE A COMPLETE')
print('Steps 1-8 done with real CADD scores')
print('RF AUC=0.904 | XGB AUC=0.906')
print('Next: Phase B or paper writing')